<a href="https://kaggle.com/kernels/welcome?src=https://github.com/Mahendra2409/Blender/blob/main/Kaggel_Script/blender-render-farm-kaggle.ipynb" target="_parent"><img src="https://img.shields.io/badge/Open_in-Kaggle-20BEFF?logo=kaggle&logoColor=white" alt="Open In Kaggle"/></a>
<a href="https://colab.research.google.com/github/Mahendra2409/Blender/blob/main/Kaggel_Script/blender-render-farm-kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Blender Render Farm — Kaggle GPU

Render pre-made `.blend` animation files on Kaggle's free GPU.

**How it works:**
1. You specify the `.blend` filename in the config cell below
2. The script auto-detects which Blender version the file needs
3. Downloads the matching Blender build + your `.blend` from Google Drive
4. Renders the animation with **zero changes** to render settings
5. Equivalent to opening the file and clicking **Render → Render Animation**

> **Requirements:** Enable **GPU accelerator** (T4 ×2) and **Internet access** in Kaggle settings.

In [ ]:
# @title 📋 1. Config — Change BLEND_FILENAME to render a different file

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  USER CONFIG — Edit only this section
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BLEND_FILENAME = "Geo_Falling_Cubes_Animation.blend"

GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1ilDk8ADzm2_OM2B5--kVFWRtq_tmotZq?usp=sharing"

OUTPUT_DIR = "/kaggle/working/renders"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f"🎬 Will render: {BLEND_FILENAME}")
print(f"📂 From Drive:  {GDRIVE_FOLDER_URL}")
print(f"💾 Output to:   {OUTPUT_DIR}")

In [ ]:
# @title 📥 2. Download .blend from Google Drive
import subprocess, os, shutil

subprocess.check_call(["pip", "install", "-q", "gdown"])
import gdown

BLEND_DIR = "/kaggle/working/blend_files"
os.makedirs(BLEND_DIR, exist_ok=True)

# Download the entire public folder
print(f"📥 Downloading from Google Drive folder...")
temp_dl = "/kaggle/working/_gdrive_download"
gdown.download_folder(
    url=GDRIVE_FOLDER_URL,
    output=temp_dl,
    quiet=False,
    remaining_ok=True,
)

# Find and move the target .blend file
blend_path = None
for root, dirs, files in os.walk(temp_dl):
    for fname in files:
        src = os.path.join(root, fname)
        dest = os.path.join(BLEND_DIR, fname)
        shutil.move(src, dest)
        print(f"  Moved: {fname}")
        if fname == BLEND_FILENAME:
            blend_path = dest

# Cleanup temp dir
shutil.rmtree(temp_dl, ignore_errors=True)

if blend_path and os.path.exists(blend_path):
    size_mb = os.path.getsize(blend_path) / (1024 * 1024)
    print(f"\n✅ Found: {BLEND_FILENAME} ({size_mb:.1f} MB)")
    print(f"   Path:  {blend_path}")
else:
    available = [f for f in os.listdir(BLEND_DIR) if f.endswith('.blend')]
    print(f"\n❌ '{BLEND_FILENAME}' not found in the Drive folder!")
    print(f"   Available .blend files: {available}")
    raise FileNotFoundError(f"'{BLEND_FILENAME}' not found. Check the filename.")

In [ ]:
# @title 🔍 3. Detect Blender Version & Install Matching Build
import gzip, re, struct, os, subprocess

def detect_blend_version(filepath):
    """Detect Blender version from .blend file header.
    
    Supports uncompressed, gzip, and zstd-compressed .blend files.
    Returns (major, minor) tuple.
    
    Header formats:
      Legacy (pre-4.x): BLENDER[_-][vV]XXX  (XXX = version digits, e.g. '306' = 3.6)
      Modern (4.x+):    BLENDER...vMMmm...   (MM = major, mm = minor, e.g. 'v0502' = 5.2)
    """
    with open(filepath, 'rb') as f:
        magic = f.read(4)
    
    # Decompress header
    if magic == b'\x28\xb5\x2f\xfd':  # Zstandard
        try:
            import zstandard as zstd
        except ImportError:
            import sys
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'zstandard', '-q'])
            import zstandard as zstd
        dctx = zstd.ZstdDecompressor()
        with open(filepath, 'rb') as f:
            raw = dctx.stream_reader(f).read(24)
    elif magic[:2] == b'\x1f\x8b':  # Gzip
        with gzip.open(filepath, 'rb') as gz:
            raw = gz.read(24)
    else:  # Uncompressed
        with open(filepath, 'rb') as f:
            raw = f.read(24)
    
    header_str = raw[:24].decode('ascii', errors='replace')
    assert header_str.startswith('BLENDER'), f'Not a valid .blend file!'
    
    # Modern format (Blender 4.x+): look for vMMmm pattern
    match = re.search(r'v(\d{4})', header_str)
    if match:
        code = match.group(1)
        major = int(code[:2])  # e.g., '05' -> 5
        minor = int(code[2:])  # e.g., '02' -> 2
        return (major, minor)
    
    # Legacy format: bytes 9-11 are 3 ASCII digits
    try:
        version_str = raw[9:12].decode('ascii')
        if version_str.isdigit():
            v = int(version_str)
            return (v // 100, (v % 100) // 10)
    except (UnicodeDecodeError, ValueError):
        pass
    
    raise ValueError(f'Could not detect version from header: {header_str}')


def find_best_download(major, minor):
    """Scrape Blender download page to find the latest patch for major.minor."""
    import urllib.request
    folder_url = f'https://download.blender.org/release/Blender{major}.{minor}/'
    print(f"  Checking: {folder_url}")
    
    try:
        with urllib.request.urlopen(folder_url) as resp:
            html = resp.read().decode('utf-8')
    except Exception as e:
        raise RuntimeError(f'Could not access {folder_url}: {e}')
    
    # Find all linux-x64 tar.xz files
    pattern = rf'blender-({major}\.{minor}\.\d+)-linux-x64\.tar\.xz'
    matches = re.findall(pattern, html)
    
    if not matches:
        raise RuntimeError(f'No linux-x64 builds found for Blender {major}.{minor}!')
    
    # Sort by patch version and pick the latest
    matches.sort(key=lambda v: [int(x) for x in v.split('.')])
    best_version = matches[-1]
    filename = f'blender-{best_version}-linux-x64.tar.xz'
    url = folder_url + filename
    return url, best_version, filename


# --- Detect version ---
print(f"🔍 Detecting Blender version from: {BLEND_FILENAME}")
major, minor = detect_blend_version(blend_path)
print(f"   Detected: Blender {major}.{minor}.x")

# --- Find best download ---
print(f"\n🌐 Finding latest Blender {major}.{minor} build...")
download_url, version_str, tar_filename = find_best_download(major, minor)
print(f"   Best match: Blender {version_str}")
print(f"   URL: {download_url}")

# --- Download & extract ---
blender_dir = tar_filename.replace('.tar.xz', '')
blender_exe = f'./{blender_dir}/blender'

if os.path.exists(blender_exe):
    print(f"\n✅ Blender already installed: {blender_exe}")
else:
    print(f"\n📥 Downloading Blender {version_str}...")
    !wget -nc -q {download_url}
    print(f"📦 Extracting...")
    !tar -xf {tar_filename}
    print(f"✅ Blender {version_str} ready: {blender_exe}")

# Save for later cells
BLENDER_EXE = blender_exe
BLENDER_VERSION = version_str
BLEND_PATH = blend_path

In [ ]:
%%writefile render_blend.py

"""
Blender Render Farm — Core Render Script

Opens a .blend file and renders its animation with ZERO changes
to render settings. Only GPU config and output path are modified.

Equivalent to: Open file → Render → Render Animation
"""
import bpy
import os
import sys
import time


# ============================================================
#  Read config from environment variables (set by notebook)
# ============================================================
BLEND_FILE = os.environ.get("BLEND_FILE", "")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working/renders")

if not BLEND_FILE:
    print("ERROR: BLEND_FILE environment variable not set!")
    sys.exit(1)


# ============================================================
#  GPU Setup — The ONLY render setting we touch
# ============================================================
def setup_gpu():
    """Configure GPU rendering. Tries OptiX → CUDA → HIP → CPU."""
    scene = bpy.context.scene

    if scene.render.engine != 'CYCLES':
        print(f"  Engine is {scene.render.engine} (not Cycles). Skipping GPU setup.")
        return

    scene.cycles.device = 'GPU'
    prefs = bpy.context.preferences.addons['cycles'].preferences

    for compute_type in ['OPTIX', 'CUDA', 'HIP', 'METAL', 'ONEAPI']:
        try:
            prefs.compute_device_type = compute_type
            prefs.get_devices()
            gpu_found = False
            for device in prefs.devices:
                if device.type != 'CPU':
                    device.use = True
                    gpu_found = True
                    print(f"  ✅ Enabled: {device.name} ({device.type})")
                else:
                    device.use = False
            if gpu_found:
                print(f"  Compute type: {compute_type}")
                return
        except Exception:
            continue

    print("  ⚠️ No GPU found, falling back to CPU rendering.")
    scene.cycles.device = 'CPU'


# ============================================================
#  Main
# ============================================================
def render_blend():
    print(f"\n{'='*60}")
    print(f"  BLENDER RENDER FARM")
    print(f"{'='*60}")

    # --- Step 1: Open the .blend file ---
    print(f"\n📂 Opening: {BLEND_FILE}")
    bpy.ops.wm.open_mainfile(filepath=BLEND_FILE)
    scene = bpy.context.scene

    # --- Step 2: Log the render settings (read-only, NOT modified) ---
    print(f"\n--- Render Settings (FROM BLEND FILE — NOT MODIFIED) ---")
    print(f"  Engine:      {scene.render.engine}")
    if scene.render.engine == 'CYCLES':
        print(f"  Samples:     {scene.cycles.samples}")
        print(f"  Denoising:   {scene.cycles.use_denoising}")
        if scene.cycles.use_denoising:
            print(f"  Denoiser:    {scene.cycles.denoiser}")
    print(f"  Resolution:  {scene.render.resolution_x}×{scene.render.resolution_y} @ {scene.render.resolution_percentage}%")
    print(f"  Frames:      {scene.frame_start} → {scene.frame_end} (step {scene.frame_step})")
    print(f"  Format:      {scene.render.image_settings.file_format}")

    total_frames = (scene.frame_end - scene.frame_start) // scene.frame_step + 1
    print(f"  Total:       {total_frames} frame(s)")

    # --- Step 3: Setup GPU (ONLY modification) ---
    print(f"\n--- GPU Setup ---")
    setup_gpu()

    # --- Step 4: Set output path (ONLY other modification) ---
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    scene.render.filepath = os.path.join(OUTPUT_DIR, "frame_")
    print(f"\n  Output path: {scene.render.filepath}")

    # --- Step 5: Render animation ---
    print(f"\n{'='*60}")
    print(f"  🚀 STARTING RENDER — {total_frames} frame(s)")
    print(f"{'='*60}\n")

    start_time = time.time()
    bpy.ops.render.render(animation=True)
    elapsed = time.time() - start_time

    # --- Step 6: Summary ---
    print(f"\n{'='*60}")
    print(f"  ✅ RENDER COMPLETE!")
    print(f"  Total time:    {elapsed:.1f}s ({elapsed/60:.1f} min)")
    if total_frames > 1:
        print(f"  Avg per frame: {elapsed/total_frames:.1f}s")
    print(f"  Output dir:    {OUTPUT_DIR}")
    print(f"{'='*60}\n")

    # List rendered files
    if os.path.exists(OUTPUT_DIR):
        rendered = sorted([f for f in os.listdir(OUTPUT_DIR) if not f.startswith('.')])
        print(f"  Rendered files ({len(rendered)}):")
        for f in rendered:
            size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
            print(f"    {f} ({size_kb:.0f} KB)")


if __name__ == "__main__":
    render_blend()

In [ ]:
# @title 🚀 5. Start Rendering
import os

# Pass config to Blender via environment variables
os.environ["BLEND_FILE"] = BLEND_PATH
os.environ["OUTPUT_DIR"] = OUTPUT_DIR

print(f"🎬 Rendering: {BLEND_FILENAME}")
print(f"🖥️  Blender:   {BLENDER_VERSION}")
print(f"💾 Output:    {OUTPUT_DIR}")
print(f"\n{'—'*50}\n")

!{BLENDER_EXE} -b -P render_blend.py

In [ ]:
# @title 📦 6. Package Results for Download
import os, zipfile

# List rendered files
if os.path.exists(OUTPUT_DIR):
    rendered = sorted([f for f in os.listdir(OUTPUT_DIR) if not f.startswith('.')])
    total_size = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in rendered)
    
    print(f"📊 Render Summary")
    print(f"   Files:      {len(rendered)}")
    print(f"   Total size: {total_size / (1024*1024):.1f} MB")
    print(f"\n   Files:")
    for f in rendered:
        size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
        print(f"     {f} ({size_kb:.0f} KB)")
    
    # Create zip
    blend_name = BLEND_FILENAME.replace('.blend', '')
    zip_path = f"/kaggle/working/{blend_name}_rendered_frames.zip"
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in rendered:
            zf.write(os.path.join(OUTPUT_DIR, f), f)
    
    zip_size = os.path.getsize(zip_path) / (1024*1024)
    print(f"\n✅ Packaged: {zip_path} ({zip_size:.1f} MB)")
    print(f"   Download from Kaggle Output tab →")
else:
    print(f"❌ No renders found in {OUTPUT_DIR}")